<a href="https://colab.research.google.com/github/juhungaro/Pipoca/blob/main/DadosEarthEngine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ee
import pandas as pd


# Autenticar e inicializar o Earth Engine
ee.Authenticate()
ee.Initialize(project="pipoca-452815")

# Definir região de interesse (Sorriso, Mato Grosso)
sorriso = ee.Geometry.Point([-55.71, -12.55])  # Coordenadas aproximadas

# Definir o período de tempo
start_date = '2015-01-01'
end_date = '2024-03-31'

# Criar uma lista de todos os meses no período
def generate_monthly_dates(start, end):
    start_date = pd.to_datetime(start)
    end_date = pd.to_datetime(end)
    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    # Removed getInfo() call and directly create ee.Date objects
    ee_dates = ee.List(dates.strftime('%Y-%m-%d').tolist()).map(lambda date_str: ee.Date(date_str))
    # Use get('year') and get('month') within a server-side function
    # Corrected to directly create FeatureCollection from the mapped list
    return ee.FeatureCollection(ee_dates.map(lambda date: ee.Feature(None, {'year': ee.Date(date).get('year'), 'month': ee.Date(date).get('month')})))


# Coleção de imagens Landsat (exemplo, escolha a coleção que melhor se adapta à sua necessidade)
landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
    .filterBounds(sorriso) \
    .filterDate(start_date, end_date)

# Função para calcular o NDVI
def calculate_ndvi(image):
    nir = image.select('SR_B5')  # Banda NIR para Landsat 8
    red = image.select('SR_B4')  # Banda Vermelha para Landsat 8
    ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')
    return image.addBands(ndvi)

# Aplica a função para calcular o NDVI em toda a coleção
ndvi_collection = landsat.map(calculate_ndvi)

# Função para reduzir a imagem para a média do NDVI na região
def reduce_monthly_ndvi(year_month):
    year = ee.Number(year_month.get('year'))
    month = ee.Number(year_month.get('month'))
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')

    monthly_image = ndvi_collection.filterDate(start, end).mean()

    if monthly_image:
        mean_ndvi = monthly_image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=sorriso,
            scale=30,  # Ajuste a escala conforme a resolução da imagem
            maxPixels=1e9
        ).get('NDVI')
        return ee.Feature(None, {'year': year, 'month': month, 'mean_ndvi': mean_ndvi})
    else:
        return ee.Feature(None, {'year': year, 'month': month, 'mean_ndvi': None})

monthly_dates = generate_monthly_dates(start_date, end_date)

# Mapeamento da função de redução sobre a lista de meses
monthly_ndvi_features = monthly_dates.map(reduce_monthly_ndvi) # Modified: monthly_dates is already a FeatureCollection

# Converter os resultados do Earth Engine para um DataFrame pandas
def ee_to_pandas(data):
    features = data.getInfo()['features']
    dict_list = []
    for f in features:
        dict_list.append(f['properties'])
    return pd.DataFrame(dict_list)

# Obter os resultados e converção para pandas
monthly_ndvi_df = ee_to_pandas(monthly_ndvi_features)

# Imprimit o dataframe resultante
print(monthly_ndvi_df)

# Salvar o dataframe em  CSV
monthly_ndvi_df.to_csv('mndvi_mensal_sorriso.csv', index=False)

     mean_ndvi  month  year
0     0.077605      1  2015
1     0.097902      2  2015
2     0.042416      3  2015
3     0.082564      4  2015
4     0.069358      5  2015
..         ...    ...   ...
106   0.046528     11  2023
107   0.103179     12  2023
108   0.047424      1  2024
109   0.047743      2  2024
110   0.076415      3  2024

[111 rows x 3 columns]
